In [47]:
import pandas as pd 
import numpy as np
from pandas.api.types import CategoricalDtype

In [30]:
filepath="Dataset.csv"
df=pd.read_csv(filepath,index_col=False)

In [31]:
df.head(4)

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly


In [33]:
df.shape

(3900, 18)

In [34]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [ ]:
df['Frequency of Purchases'] = df['Frequency of Purchases'].str.strip()
df['Season'] = df['Season'].str.strip()
print("Cleaned Frequencies:", df['Frequency of Purchases'].unique())

Cleaned Frequencies: ['Fortnightly' 'Weekly' 'Annually' 'Quarterly' 'Bi-Weekly' 'Monthly'
 'Every 3 Months']


In [ ]:
null_ratings = df[df['Review Rating'].isnull()]
null_ratings

38      44
50      39
80      24
96      43
262      6
330     43
356     27
383      6
419     14
442     31
452     39
460     27
491     24
512      2
560     12
628     29
799     31
829     14
861     35
888     44
1072    34
1109     2
1124    40
1134     2
1151    24
1176    39
1189    28
1208     8
1220    36
1230    49
1247    18
1292    42
1313    23
1326    17
1334    12
1341    31
1370    41
Name: Previous Purchases, dtype: int64

For filling the empty reviews i have assigned them the median but with the condition if the person has purchased more than 20 times you will give some extra points as he/she is a frequent buyer

In [41]:
df['Review Rating'] = df.groupby(['Category', 'Gender'])['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)
high_tenure_mask = (df['Review Rating'].isnull()) & (df['Previous Purchases'] >= 20)
df.loc[high_tenure_mask, 'Review Rating'] = np.clip(df.loc[high_tenure_mask, 'Review Rating'] + 0.5, 2.5, 5.0)

ONE HOT ENCODING ON THESE COLUMNS

In [42]:
binary_columns = ['Discount Applied', 'Promo Code Used', 'Subscription Status']

for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [49]:
df.columns.unique()

Index(['Customer ID', 'Age', 'Gender', 'Item Purchased', 'Category',
       'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season',
       'Review Rating', 'Subscription Status', 'Shipping Type',
       'Discount Applied', 'Promo Code Used', 'Previous Purchases',
       'Payment Method', 'Frequency of Purchases'],
      dtype='object')

In [45]:
from pandas.api.types import CategoricalDtype
season_order = CategoricalDtype(
    categories=['Spring', 'Summer', 'Fall', 'Winter'], 
    ordered=True
)
df['Season'] = df['Season'].astype(season_order)

In [50]:
frequency_map = {
    "Weekly": 52,
    "Fortnightly": 26,
    "Bi-Weekly": 26,
    "Monthly": 12,
    "Every 3 Months": 4,
    "Quarterly": 4,
    "Annually": 1
}
df['purchases_per_year'] = df['Frequency of Purchases'].map(frequency_map)

What it does: It estimates long-term revenue generation by multiplying three things: how much they spent today, how many times they buy per year, and their relationship length.The Math Trick: Since the maximum number of Previous Purchases in the dataset is $50$, dividing their history by $50$ creates a decimal scale from $0.02$ to $1.0$. This penalizes brand-new switchers and heavily rewards your long-term, veteran shoppers.

In [53]:
df['pdr'] = df['Promo Code Used']
df['eltv'] = df['Purchase Amount (USD)'] * df['purchases_per_year'] * (df['Previous Purchases'] / 50)

In [54]:
# 3. Spend Efficiency Score (Category-Level Premium)
avg_full = df[df['Discount Applied'] == 0].groupby('Category')['Purchase Amount (USD)'].mean()
avg_disc = df[df['Discount Applied'] == 1].groupby('Category')['Purchase Amount (USD)'].mean()
category_efficiency = avg_full - avg_disc

# Map the efficiency score back to each customer based on what category they bought
df['spend_efficiency'] = df['Category'].map(category_efficiency)

In [55]:
# 4. Satisfaction Index
max_freq = df['purchases_per_year'].max()
df['satisfaction_index'] = (df['Review Rating'] / 5) * (df['purchases_per_year'] / max_freq)

In [56]:
# 5. Payment Commitment Score
# Credit/Debit Card = 2 pts, PayPal/Venmo = 1 pt, Cash = 0 pts
payment_map = {
    'Credit Card': 3, 'Debit Card': 2,
    'PayPal': 1, 'Venmo': 1,
    'Cash': 0
}
df['payment_score'] = df['Payment Method'].map(payment_map)

In [57]:
df['R_score'] = 6 - pd.qcut(df['Previous Purchases'], 5, labels=False, duplicates='drop') - 1
df['F_score'] = pd.qcut(df['purchases_per_year'], 5, labels=False, duplicates='drop') + 1
df['M_score'] = pd.qcut(df['Purchase Amount (USD)'], 5, labels=False, duplicates='drop') + 1
df['rfm_score'] = df['R_score'] + df['F_score'] + df['M_score']

In [58]:
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,...,purchases_per_year,pdr,eltv,spend_efficiency,satisfaction_index,payment_score,R_score,F_score,M_score,rfm_score
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,26,1,385.84,0.471264,0.310000,1.0,4,2,3,9
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,26,1,66.56,0.471264,0.310000,0.0,5,2,3,10
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,52,1,1746.16,0.471264,0.620000,3.0,3,3,4,10
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,52,1,4586.40,-2.719816,0.700000,1.0,1,3,5,9
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,1,1,30.38,0.471264,0.010385,1.0,2,1,2,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,...,52,0,931.84,0.471264,0.840000,1.0,2,3,1,6
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,...,26,0,1044.68,2.399655,0.450000,NaN,1,2,2,5
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,...,4,0,63.36,2.399655,0.044615,1.0,3,1,1,5
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,...,52,0,1921.92,-2.719816,0.760000,1.0,3,3,4,10
